# Assignment 3 - Logan Bolton

# Word Embeddings

In [5]:
import gensim.downloader as api
wv = api.load('glove-twitter-50')
import numpy as np

## 1a

In [12]:
words = ['dog', 'bark', 'tree', 'bank', 'river', 'money']
n = len(words)

sim_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        sim_matrix[i][j] = wv.similarity(words[i], words[j])
 
header = f"{'':>8}" + "".join(f"{w:>8}" for w in words)
print(header)
for i in range(n):
    row = f"{words[i]:>8}" + "".join(f"{glove_sim[i][j]:>8.4f}" for j in range(n))
    print(row)

             dog    bark    tree    bank   river   money
     dog  1.0000  0.5938  0.7138  0.3482  0.4012  0.5751
    bark  0.5938  1.0000  0.5459  0.0401  0.2666  0.2910
    tree  0.7138  0.5459  1.0000  0.3495  0.4871  0.5101
    bank  0.3482  0.0401  0.3495  1.0000  0.3199  0.6747
   river  0.4012  0.2666  0.4871  0.3199  1.0000  0.3378
   money  0.5751  0.2910  0.5101  0.6747  0.3378  1.0000


## 1b

In [15]:
from gensim.models import FastText
from gensim.test.utils import common_texts
ft = FastText(vector_size=50, window=5, min_count=1, sentences=common_texts, epochs=10)
 
sim_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        sim_matrix[i][j] = ft.wv.similarity(words[i], words[j])
 
header = f"{'':>8}" + "".join(f"{w:>8}" for w in words)
print(header)
for i in range(n):
    row = f"{words[i]:>8}" + "".join(f"{sim_matrix[i][j]:>8.4f}" for j in range(n))
    print(row)
 


             dog    bark    tree    bank   river   money
     dog  1.0000  0.1078 -0.1700  0.0313 -0.0135 -0.1112
    bark  0.1078  1.0000  0.2076  0.1696  0.0867 -0.0468
    tree -0.1700  0.2076  1.0000  0.0357  0.0653 -0.2631
    bank  0.0313  0.1696  0.0357  1.0000  0.2033 -0.0164
   river -0.0135  0.0867  0.0653  0.2033  1.0000 -0.1227
   money -0.1112 -0.0468 -0.2631 -0.0164 -0.1227  1.0000


## 1c

I think GloVE captured the word embeddings more accurately. FastText has very low similarity between most words, even in cases like bank and money (-0.0164) while GloVE captures it more accurately with 0.6747.

# LSTM

In [19]:
"""Minimal LSTM for tweet emoji prediction using TweetEval."""

import re
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from datasets import load_dataset
from collections import Counter

# ── Config ───────────────────────────────────────────────────────────────────
VOCAB_SIZE = 15000
EMBED_DIM = 64
HIDDEN_DIM = 128
MAX_LEN = 40
BATCH_SIZE = 256
EPOCHS = 8
NUM_CLASSES = 20
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ── 1. Load data ─────────────────────────────────────────────────────────────
ds = load_dataset("cardiffnlp/tweet_eval", "emoji", split="train")
texts, labels = ds["text"], ds["label"]

# ── 2. Preprocess ────────────────────────────────────────────────────────────

# (a) Clean tweets: remove URLs, @mentions, #hashtags, special characters
def clean(text):
    text = re.sub(r"http\S+|www\S+", "", text)   # URLs
    text = re.sub(r"@\w+", "", text)              # mentions
    text = re.sub(r"#\w+", "", text)              # hashtags
    text = re.sub(r"[^a-zA-Z\s]", "", text)       # special characters
    return text.lower().strip()

texts = [clean(t) for t in texts]

# (b) Tokenize and pad sequences
counter = Counter(w for t in texts for w in t.split())
vocab = {w: i+2 for i, (w, _) in enumerate(counter.most_common(VOCAB_SIZE))}
# 0 = pad, 1 = unk

def encode(text):
    ids = [vocab.get(w, 1) for w in text.split()][:MAX_LEN]
    return ids + [0] * (MAX_LEN - len(ids))

X = torch.tensor([encode(t) for t in texts])

# (c) One-hot encode labels
y_onehot = F.one_hot(torch.tensor(labels), num_classes=NUM_CLASSES).float()

# ── 3. Split: 80% train, 10% val, 10% test ───────────────────────────────────
n = len(X)
idx = torch.randperm(n)
X, y_onehot = X[idx], y_onehot[idx]

n_train = int(0.8 * n)
n_val = int(0.1 * n)

X_train, y_train = X[:n_train], y_onehot[:n_train]
X_val, y_val = X[n_train:n_train+n_val], y_onehot[n_train:n_train+n_val]
X_test, y_test = X[n_train+n_val:], y_onehot[n_train+n_val:]

print(f"Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}")

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=BATCH_SIZE)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=BATCH_SIZE)

# ── Model ────────────────────────────────────────────────────────────────────
class EmojiLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(VOCAB_SIZE + 2, EMBED_DIM, padding_idx=0)
        self.lstm = nn.LSTM(EMBED_DIM, HIDDEN_DIM, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(HIDDEN_DIM * 2, NUM_CLASSES)

    def forward(self, x):
        x = self.emb(x)
        _, (h, _) = self.lstm(x)
        h = torch.cat([h[-2], h[-1]], dim=1)
        return self.fc(h)

model = EmojiLSTM().to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss()  # works with one-hot labels

# ── Train ────────────────────────────────────────────────────────────────────
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        loss = loss_fn(model(xb), yb)
        opt.zero_grad()
        loss.backward()
        opt.step()
        total_loss += loss.item()

    # Validation accuracy
    model.eval()
    correct = sum(
        (model(xb.to(DEVICE)).argmax(1) == yb.to(DEVICE).argmax(1)).sum().item()
        for xb, yb in val_loader
    )
    print(f"Epoch {epoch}/{EPOCHS}  loss={total_loss/len(train_loader):.3f}  val_acc={correct/len(X_val):.3f}")

# ── Test set evaluation ──────────────────────────────────────────────────────
EMOJIS = ["❤️","😍","😂","💕","🔥","😊","😎","✨","💙","😘","📷","🇺🇸","☀️","💜","😉","💯","😁","🎄","📸","😜"]

model.eval()
correct = sum(
    (model(xb.to(DEVICE)).argmax(1) == yb.to(DEVICE).argmax(1)).sum().item()
    for xb, yb in test_loader
)
print(f"\nTest accuracy: {correct/len(X_test):.3f}  ({correct}/{len(X_test)})")

# Show a few sample predictions
print("\nSample predictions:")
for i in range(10):
    x = X_test[i].unsqueeze(0).to(DEVICE)
    pred = model(x).argmax(1).item()
    true = y_test[i].argmax(0).item()
    mark = "✓" if pred == true else "✗"
    print(f"  {mark} pred: {EMOJIS[pred]}  true: {EMOJIS[true]}")

Epoch 1/8  loss=0.227  val_acc=0.217
Epoch 2/8  loss=0.183  val_acc=0.220
Epoch 3/8  loss=0.179  val_acc=0.248
Epoch 4/8  loss=0.175  val_acc=0.259
Epoch 5/8  loss=0.172  val_acc=0.267
Epoch 6/8  loss=0.169  val_acc=0.273
Epoch 7/8  loss=0.166  val_acc=0.282
Epoch 8/8  loss=0.163  val_acc=0.284

Test accuracy: 0.270  (13502/50000)

Sample predictions:
  ✗ "en Pelham Parkway" -> pred: ❤️  true: 😂
  ✗ "The calm before...... | w/ sofarsounds @user | : B. Hall...." -> pred: 🔥  true: 📷
  ✗ "Just witnessed the great solar eclipse @ Tampa, Florida" -> pred: 🔥  true: 😎
  ✗ "This little lady is 26 weeks pregnant today! Excited for bab" -> pred: ❤️  true: 😍
  ✗ "Great road trip views! @ Shartlesville, Pennsylvania" -> pred: ❤️  true: 😁
  ✓ "CHRISTMAS DEALS BUY ANY 3 SMALL POMADES 1.5 OR 1.7 OZ RECEIV" -> pred: 🎄  true: 🎄
  ✓ "the #sisterstunt was mad real last night #MiaStaxxx #AndreaS" -> pred: 🔥  true: 🔥
  ✗ "I'm starting to love shooting in the dark #brandonwolfel @ N" -> pred: ❤️  true: 📷
  